# 01d — Ghép ba crawl run độc lập

**CPU.** Chỉ chạy sau khi cả ba thành viên hoàn thành và thư mục run của họ đều truy cập được từ runtime này. Công cụ kiểm SHA-256, đúng khoảng dòng, đúng URL/ID và cấu hình trước khi tạo run chung. Nó sao chép raw, không sửa run của thành viên. Cần dung lượng trống cho một bản sao raw.


Notebook tùy chọn, không thuộc luồng chính 00–04. Bootstrap dùng optional_data_code_lock.json riêng. Runtime mới chạy lại từ đầu để resume. Khi chạy team, cả ba ghim cùng full SHA vào CODE_REVISION trước khi crawl. Build bằng code đã merge dùng tên run mới; notebook 03 phải dùng cùng full CODE_COMMIT khi freeze.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "optional_data_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "optional_data_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Chỉ đường đến ba thư mục đã nhận

In [ ]:
import sys
sys.path.insert(0, str(CHECKOUT / "scripts"))
from team_crawl import team_plan, merge_team_crawls
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.crawl import CrawlConfig

RUN_PREFIX = "full-team-v1"  # Phải giống cả ba người
SNAPSHOT = load_snapshot(DATA_ROOT)
OFFICIAL_LINKS = parquet_path(DATA_ROOT, "links_corpus.parquet")
CONFIG = CrawlConfig(**PIPELINE_CONFIG["crawler"])
PLAN = team_plan(OFFICIAL_LINKS, workers=3, shard_size=CONFIG.shard_size, run_prefix=RUN_PREFIX)

# Đổi ba đường dẫn này thành nơi người ghép nhìn thấy các thư mục run đã nhận.
TRANSFER_ROOT = DATA_ROOT / "team_uploads"
SOURCE_DIRS = [TRANSFER_ROOT / part["run_name"] for part in PLAN["parts"]]
for part, source in zip(PLAN["parts"], SOURCE_DIRS):
    print("Người", part["member"], part["start_row"], part["stop_row"], source, "exists:", source.is_dir())
print("Plan hash:", PLAN["plan_sha256"])
print("Run sau ghép:", PLAN["merged_run_name"])


## 2. Kiểm và ghép

Lệnh sẽ từ chối nếu thiếu một phần, sai snapshot, khác code/cấu hình, mất shard, trùng khoảng hoặc raw bị đổi. Nếu dừng giữa lúc sao chép, thư mục `.staging` còn lại để chẩn đoán; không coi đó là run đã ghép.


In [ ]:
MERGED = merge_team_crawls(
    PLAN, links_path=OFFICIAL_LINKS,
    origin_corpus_sha256=SNAPSHOT["files"]["links_corpus.parquet"]["sha256"],
    source_dirs=SOURCE_DIRS, output_root=DATA_ROOT / "crawl",
)
print("Đã ghép:", MERGED["run_dir"])
print("Đủ toàn bộ dòng:", MERGED["range_complete"], MERGED["recorded_documents"], "/", PLAN["total_rows"])


## Tiếp tục pipeline

Sau khi ghép thành công, **một người** chạy notebook 02 với `CRAWL_RUN = PLAN["merged_run_name"]` (ví dụ `full-team-v1-merged`) và `BUILD_RUN` mới; rồi notebook 03 trỏ tới đúng hai tên đó. Không chạy notebook 02/03 trên từng phần riêng.
